# Weapon Detection - Module 1
### Phases 2-5: Environment Setup, Dataset Download, Verification, Visualization

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine for Phase 1 baseline).

This notebook clones the project repo structure into `/content/weapon-detection`, sets up the environment, downloads the Roboflow Pistols dataset, verifies it, and visualizes samples. Training (Phase 6) is in a separate notebook cell block appended below once this phase is confirmed working, per the phase-gated plan.

## Phase 2 - Environment Setup

In [ ]:
# Check GPU is attached
!nvidia-smi


**Design**: code lives in Git (this repo) and is cloned fresh each session -- small, fast, versioned.
Large binary artifacts (the dataset and trained weights) do NOT belong in Git; they persist in
Google Drive instead, and are symlinked into the cloned repo so scripts see them at the normal
`data/` and `runs/` paths without any path changes. This is the same repo layout you'd clone
on a laptop later -- only this Drive-symlink step is Colab-specific.

In [ ]:
# Mount Google Drive - used ONLY for persisting data/ and runs/ across sessions
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PERSIST_DIR = '/content/drive/MyDrive/weapon-detection-storage'


In [ ]:
# Set this to your GitHub repo URL (see README.md for how to create it)
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/weapon-detection.git'  # <-- EDIT ME


In [ ]:
# Clone (or pull if already cloned this session) the repo
import os

if not os.path.exists('/content/weapon-detection'):
    !git clone "$GITHUB_REPO_URL" /content/weapon-detection
else:
    %cd /content/weapon-detection
    !git pull

%cd /content/weapon-detection
!ls


In [ ]:
# Create persistent storage folders in Drive (once) and symlink them into the
# repo checkout so data/ and runs/ survive across Colab sessions without
# ever being committed to Git.
import os

for sub in ['data', 'runs', 'outputs']:
    os.makedirs(f'{DRIVE_PERSIST_DIR}/{sub}', exist_ok=True)

!rm -rf /content/weapon-detection/data /content/weapon-detection/runs /content/weapon-detection/outputs
!ln -s "$DRIVE_PERSIST_DIR/data" /content/weapon-detection/data
!ln -s "$DRIVE_PERSIST_DIR/runs" /content/weapon-detection/runs
!ln -s "$DRIVE_PERSIST_DIR/outputs" /content/weapon-detection/outputs
!ls -la /content/weapon-detection


In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

import ultralytics
ultralytics.checks()


## Phase 3 - Dataset Download

In [ ]:
# Get a free key at https://app.roboflow.com/settings/api (Private API Key)
import getpass
ROBOFLOW_API_KEY = getpass.getpass('Enter your Roboflow API key: ')


In [ ]:
!python scripts/download_dataset.py --api-key "$ROBOFLOW_API_KEY"


## Phase 4 - Dataset Verification

In [ ]:
!python scripts/verify_dataset.py


**Stop here if issues are reported above.** A handful of missing/corrupt files can be safely deleted (find them by name and `rm` the offending image/label pair). Widespread issues mean the download itself failed and you should re-run Phase 3.

## Phase 5 - Dataset Visualization

In [ ]:
!python scripts/visualize_dataset.py --split train --num-samples 9


In [ ]:
from IPython.display import Image, display
display(Image(filename='outputs/metrics/sample_annotations_train.png'))
display(Image(filename='outputs/metrics/class_distribution.png'))


---
### Checkpoint
If the sample grid shows correct bounding boxes around pistols and the class distribution chart looks reasonable (train having the most instances, valid/test smaller), Phases 1-5 are confirmed working.

**Phase 6 (Training) will be appended to this notebook next** so it inherits the same environment and mounted Drive, once you confirm this checkpoint passes on your end.